# Lab 2: Getting Data In, and Judging It

**DSA 405 · Week 2**

| | |
|---|---|
| **In class** | Friday, Aug 28 |
| **A2 due** | Thursday, Sep 3, 11:59 PM |
| **Also due Thu Sep 3** | **P1** Framing a Data Problem (separate handout) |
| **Files** | `wolfpack_dining_raw.csv`, `nc_schools_dirty.xlsx`, `permits_raleigh.json` |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Loading a file into `pandas` takes one line of code. This week we take a look at the
choices that one line makes for you by default (column types, headers, which values
count as missing), and at how to check those choices before you rely on the result.

The main file through Week 4 is `wolfpack_dining_raw.csv`: 366 inspection records for
campus dining locations. The data is synthetic and public, and its problems are modeled
on problems in real files.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
# Part 1: Explore (in class)

## Task 1.1: First load, first inspection

A default read will almost always finish without an error message. That is the
problem: nothing about an error-free read tells you the file was interpreted the way
you expected, and this one was not.

In [ ]:
dining = load("wolfpack_dining_raw.csv")

print(dining.shape)
dining.head()

366 rows, 10 columns, and `.head()` shows nothing unusual. Next, check the **dtype**
of each column. The dtype is the data type `pandas` assigned to the column when it
read the file, such as a number type or `object` (text):

In [ ]:
dining.dtypes

Nine of the ten columns came back as `object`, which means `pandas` stored them as
strings (text). Columns like `score` and `seats` should be numeric, but somewhere in
each of them there is text, so `pandas` left the whole column as strings. That result
is useful information: it tells you those columns contain values that are not numbers.

The one column that did convert to numbers, `unit_code`, should not have converted:
unit codes are labels, not quantities. Converting them to numbers causes a real
problem:

In [ ]:
as_text = load("wolfpack_dining_raw.csv", dtype=str)

lost = as_text.unit_code.str.startswith("0").sum()
print(f"unit codes that begin with 0: {lost} of {len(as_text)}")
print("as loaded by default:", dining.unit_code.head(3).tolist())
print("as they are in the file:", as_text.unit_code.head(3).tolist())

31 of 366 codes begin with a zero, and the default read removed that zero: `0352`
became `352`, which is a different label. If you later joined this table to another
table on `unit_code`, those 31 rows would fail to match, and you would see no error
message. The fix is one argument at load time: `dtype={"unit_code": str}`.

## Task 1.2: The profiling methods

**Profiling** means running a few quick checks to learn what a file contains before
you rely on it. Four methods cover most of what you can learn about a file you have
just loaded. Run each one and read the full output:

In [ ]:
dining.info()

In [ ]:
# dropna=False makes value_counts show NaN (missing) values as a row of their own
print(dining.category.value_counts(dropna=False).head(10))
print()
print("distinct category strings:", dining.category.nunique())

27 distinct category strings. Read the full list and estimate how many real categories
it represents. (`Coffee`, `COFFEE`, `coffee `, and `Coffee Shop` are variants of the
same value; in Week 4 we learn how to repair this.)

Next, profile a column that should be numeric:

In [ ]:
print(dining.score.value_counts(dropna=False).head(12))

The column contains numbers, and it also contains several values that are not numbers.
`.isna()` finds almost none of them, because most of the missing values are written as
text instead of `NaN`. You will work on this directly in A2 Task 2.2.

## Task 1.3: Excel and JSON

CSV is the simplest case. Excel files are often formatted to look nice for human
readers, and a JSON file stores data as a nested structure (lists and dictionaries
inside each other) rather than as rows. Each format needs one extra decision when you
load it.

In [ ]:
# the default read, with no extra arguments
schools_naive = load("nc_schools_dirty.xlsx", "excel")
schools_naive.head()

The sheet is not yet a table: three rows of title text appear above the real headers.
Use the `header` argument to give `read_excel` the row where the real column names
start:

In [ ]:
schools = load("nc_schools_dirty.xlsx", "excel", header=3)
print(schools.shape)
schools.head(3)

In [ ]:
# and it is THREE sheets, not one. sheet_name=None returns a dict of DataFrames.
all_sheets = load("nc_schools_dirty.xlsx", "excel", header=3, sheet_name=None)
for name, df in all_sheets.items():
    print(f"{name}: {df.shape}")

In [ ]:
# JSON: the file loads as nested lists and dictionaries, not as a table
permits = load("permits_raleigh.json", "json")
print(type(permits), "with keys:", list(permits.keys()))
print("records:", len(permits["results"]))
permits["results"][0]

The result is a dictionary that contains a list of dictionaries. `pandas` can flatten
it into a table, but choosing which fields to keep, and from which level, is a
decision you have to make; Week 11 covers it properly. For now, remember that a JSON
file loads as a nested structure, and turning it into rows is your job.

---
## Checkpoint: submit before leaving class

1. What did the default read do to `unit_code`, and exactly how many rows does it affect?
2. How many distinct `category` strings are there, and how many real categories do they
   appear to represent?
3. Name one column of `wolfpack_dining_raw.csv` that you cannot trust yet, and explain
   why.

*Answers here.*

---
# Part 2: A2 (Loading & Profiling)

This part is graded. It has four tasks.

## Task 2.1: The dtype audit

Load the dining file with default settings and audit what came back.

1. Report each column's dtype. State which columns should be numeric but are not, and
   which column converted but should not have.
2. Two columns have far more distinct values than they should. Name both columns, give
   the counts, and show two or three example values that explain why the counts are so
   high.

In [ ]:
# your audit

*Name the two columns, give the counts, and explain what makes the counts so high.*

## Task 2.2: The missing-value census

`score`, `seats`, and `avg_ticket` all contain missing values, but almost none of them
are written as `NaN`. Using `value_counts(dropna=False)` on the **string** version of
each column (`dtype=str`), build a census: a complete list of every distinct way "no
value" is written in these columns, and how many times each one appears.

Then run `pd.to_numeric(..., errors="coerce")` on each column. A **sentinel** is a
special value written inside a data column to signal "no real value here." Which of
the missing-value encodings convert into real numbers instead of becoming `NaN`? Count
them. Explain in two sentences why a sentinel that converts to a real number is a
worse problem than one that becomes `NaN`. (Consider what `.mean()` does with each.)

In [ ]:
# your census

*Census and two sentences here.*

## Task 2.3: The leading-zero repair

First show the problem, then fix it:

1. Count how many `unit_code` values lose a leading zero on a default read. Show one
   before/after pair.
2. Re-load with the correct `dtype` argument and verify the count of zero-leading codes.
3. In one sentence, name one real task that would go wrong when `0352` becomes `352`.

In [ ]:
# your repair

## Task 2.4: One table from a real web page

`read_html` reads every table on a web page at once. This is the course's first
scrape: the first time we use code to collect data from a website. It is also the
first time you will see a website refuse a request. Wikipedia responds with the error
`403 Forbidden` when a script does not say who sent it, so the starter cell includes a
User-Agent header that states honestly who we are. We cover why this matters in
Week 8.

1. Pick a Wikipedia page that has a real table (a sport, a music chart, a discography)
   on a topic you know well enough to notice a wrong value. `pd.read_html(...)` returns
   a **list** of tables; find the table you want inside that list.
2. Perform **one** cleaning action the table needs (remove a row that is not an
   observation, fix a header, or convert a column) and state the row count before and
   after.
3. The most important graded part, written in prose: judge whether the table is fit
   for one specific purpose that you name. Who or what is missing from it? Who decided
   what counts as a row? A table of "every #1 hit" contains decisions that a person
   made; name one of those decisions and say who or what it leaves out.

One paragraph that names something specific is worth more than three paragraphs that
only say "the data may be incomplete." 

In [ ]:
URL = "..."   # your Wikipedia page

# Tell the site who you are. Wikipedia refuses requests from anonymous scripts.
# We identify ourselves honestly instead of pretending to be a browser; Week 8 covers why.
UA = {"User-Agent": "DSA405-student-lab/1.0 (NC State class exercise)"}

# html = requests.get(URL, headers=UA, timeout=30).text
# tables = pd.read_html(io.StringIO(html))

*Fitness-for-purpose paragraph here.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A2_[yourUnityID].ipynb`
4. Upload to the **A2** space on Moodle.

The **Checkpoint** section is submitted separately to **Week 2 In-Class Activity**, before
the end of class on Friday. Due for A2: **Thursday, Sep 3, 11:59 PM**.